# O que é o Pandera e como integrá-lo com Pandas
Introdução ao Pandera
Pandera é uma biblioteca Python para validação de dados que se integra perfeitamente com o Pandas.
Ele permite definir esquemas de validação de forma declarativa e executar verificações em DataFrames e Series.

Principais Características:

Integração com Pandas: Trabalha diretamente com DataFrames e Series

Validação Declarativa: Define regras de forma clara e legível

Tipos de Dados: Suporte a validação de tipos nativos do Python e do Pandas

Validação Personalizada: Permite criar regras customizadas

Mensagens de Erro: Fornece mensagens detalhadas sobre falhas de validação

In [ ]:
pip install pandera

In [2]:
import pandas as pd
import pandera.pandas as pa

# Criando um DataFrame de exemplo
df = pd.DataFrame({
    'idade': [25, 30, 35],
    'salario': [5000, 6000, 7000],
    'email': ['joao@email.com', 'maria@email.com', 'pedro@email.com']
})

# Definindo o esquema de validação
schema = pa.DataFrameSchema({
    'idade': pa.Column(int, checks=pa.Check.ge(0)),  # idade >= 0
    'salario': pa.Column(float, checks=pa.Check.gt(0)),  # salario > 0
    'email': pa.Column(str, checks=pa.Check.str_matches(r'^[^@]+@[^@]+\.[^@]+$'))
})

# Validando o DataFrame
try:
    schema.validate(df)
    print("Dados válidos!")
except pa.errors.SchemaError as e:
    print(f"Erro de validação: {e}")

Erro de validação: expected series 'salario' to have type float64, got int64


In [6]:
print(df.dtypes)

idade       int64
salario     int64
email      object
dtype: object


## Tipos de Validação Suportados
Validação de Colunas:

Tipo de dados
Valores nulos
Valores únicos
Valores em uma lista específica

Validação de Linhas:

Relações entre colunas
Agregações
Condições complexas

Validação de DataFrame:

Estrutura do DataFrame
Número de linhas/colunas
Índices

In [7]:
# Esquema mais complexo
schema_avancado = pa.DataFrameSchema({
    'idade': pa.Column(
        int,
        checks=[
            pa.Check.ge(0),  # Idade deve ser maior ou igual a zero
            pa.Check.le(120)  # Idade deve ser menor ou igual a 120 anos
        ],
        nullable=False  # Não permite valores nulos
    ),
    'salario': pa.Column(
        float,
        checks=[
            pa.Check.gt(0),  # Salário deve ser maior que zero
            pa.Check.lt(1000000)  # Salário deve ser menor que 1 milhão
        ]
    ),
    'email': pa.Column(
        str,
        checks=pa.Check.str_matches(r'^[^@]+@[^@]+\.[^@]+$'),  # Valida formato de email
        nullable=False  # Não permite valores nulos
    ),
    'departamento': pa.Column(
        str,
        checks=pa.Check.isin(['TI', 'RH', 'Vendas', 'Financeiro'])  # Departamento deve estar na lista de valores permitidos
    )
})

# Validação com transformação
def ajustar_salario(df):
    df = df.copy()
    df['salario'] = df['salario'] * 1.1  # Aumento de 10%
    return df

# Usando o método pipe para transformação
schema_com_transformacao = pa.DataFrameSchema({
    'salario': pa.Column(
        float,
        checks=pa.Check.gt(0),
        coerce=True
    )
})

# Exemplo de uso:
df_validado = schema_com_transformacao.validate(df) # Verifica se a coluna é valida
df_transformado = df_validado.pipe(ajustar_salario) # aplica a transformação

In [8]:
df_validado

,idade,salario,email
0,25,5000.0,joao@email.com
1,30,6000.0,maria@email.com
2,35,7000.0,pedro@email.com


In [9]:
df_transformado

,idade,salario,email
0,25,5500.0,joao@email.com
1,30,6600.0,maria@email.com
2,35,7700.0,pedro@email.com
